# GitDB - Git Database Abstraction Library

GitDB is a pure Python implementation of the Git object database. It provides low-level access to Git repositories and object storage.

## Example 1: Basic GitDB Repository Access

In [ ]:
import gitdb
import os
import tempfile

# Create a temporary directory for testing
with tempfile.TemporaryDirectory() as temp_dir:
    # Initialize a bare git repository
    git_dir = os.path.join(temp_dir, ".git")
    os.makedirs(os.path.join(git_dir, "objects"), exist_ok=True)
    os.makedirs(os.path.join(git_dir, "refs"), exist_ok=True)
    
    # Write basic git config
    with open(os.path.join(git_dir, "config"), "w") as f:
        f.write("[core]\n    bare = true\n")
    
    print(f"Created git directory: {git_dir}")
    
    # Try to access the git database
    try:
        db = gitdb.GitDB(git_dir)
        print(f"GitDB initialized successfully")
        print(f"Database root: {db.root_path()}")
    except Exception as e:
        print(f"GitDB initialization failed: {e}")
        raise

## Example 2: Working with Object Database

In [ ]:
import gitdb
from gitdb.db import LooseObjectDB
import tempfile
import os

# Create a temporary git objects directory
with tempfile.TemporaryDirectory() as temp_dir:
    objects_dir = os.path.join(temp_dir, "objects")
    os.makedirs(objects_dir, exist_ok=True)
    
    # Create a loose object database
    loose_db = LooseObjectDB(objects_dir)
    print(f"Loose object database created at: {objects_dir}")
    
    # Test database properties
    print(f"Root path: {loose_db.root_path()}")
    print(f"Database type: {type(loose_db).__name__}")
    
    # Check if database is accessible
    try:
        # Try to iterate over objects (should be empty initially)
        object_count = sum(1 for _ in loose_db.sha_iter())
        print(f"Number of objects in database: {object_count}")
    except Exception as e:
        print(f"Error accessing database: {e}")
        raise

## Example 3: Object Stream and Memory Handling

In [ ]:
import gitdb
from gitdb.stream import DecompressMemMapReader
import io

# Create a test data stream
test_data = b"Hello, GitDB! This is test content for stream processing."
stream = io.BytesIO(test_data)

print(f"Original data length: {len(test_data)}")
print(f"Original data: {test_data[:50]}...")

# Test stream properties
print(f"Stream position: {stream.tell()}")
print(f"Stream readable: {stream.readable()}")
print(f"Stream seekable: {stream.seekable()}")

# Read data back
stream.seek(0)
read_data = stream.read()
print(f"Read data matches original: {read_data == test_data}")

# Test with different chunk sizes
stream.seek(0)
chunk = stream.read(10)
print(f"First 10 bytes: {chunk}")

stream.close()

## Example 4: Git Object Types and Utilities

In [ ]:
import gitdb.typ as git_types
from gitdb.util import bin_to_hex, hex_to_bin

# Test Git object type constants
print("Git Object Types:")
print(f"Blob type: {git_types.str_blob_type}")
print(f"Tree type: {git_types.str_tree_type}")
print(f"Commit type: {git_types.str_commit_type}")
print(f"Tag type: {git_types.str_tag_type}")

# Test hex/binary conversion utilities
test_hex = "abc123def456"
test_binary = hex_to_bin(test_hex)
converted_back = bin_to_hex(test_binary)

print(f"\nHex conversion test:")
print(f"Original hex: {test_hex}")
print(f"Binary length: {len(test_binary)} bytes")
print(f"Converted back: {converted_back}")
print(f"Conversion successful: {test_hex == converted_back}")

# Test with SHA-1 like string
sha1_hex = "a" * 40  # 40 character hex string (SHA-1 length)
sha1_bin = hex_to_bin(sha1_hex)
print(f"\nSHA-1 test - Length: {len(sha1_bin)} bytes (should be 20)")

## Example 5: Error Handling and Edge Cases

In [ ]:
import gitdb
import gitdb.exc
import tempfile
import os

# Test error conditions
print("Testing GitDB error handling:")

# Test with non-existent directory: GitDB() is lazy by design (FileDBBase does no accessibility check at
# construction), so the bad root surfaces as InvalidDBRoot on the FIRST database access
invalid_db = gitdb.GitDB("/non/existent/path")
try:
    invalid_db.databases()
except gitdb.exc.InvalidDBRoot as e:
    assert "/non/existent/path" in str(e), f"unexpected message: {e!r}"
    print(f"✓ Non-existent path rejected on first access ({type(e).__name__})")
else:
    raise AssertionError("GitDB('/non/existent/path').databases() did not raise InvalidDBRoot")

# Test with empty directory: an existing directory is a valid, empty loose-object database
with tempfile.TemporaryDirectory() as temp_dir:
    empty_db = gitdb.GitDB(temp_dir)
    assert len(empty_db.databases()) == 1, f"expected one loose-object db, got {empty_db.databases()!r}"
    assert empty_db.size() == 0, f"an empty directory should hold 0 objects, got {empty_db.size()}"
    print("✓ GitDB accepts an empty directory (1 loose-object db, 0 objects)")

# Test invalid hex conversion
import binascii
from gitdb.util import hex_to_bin
try:
    invalid_result = hex_to_bin("xyz123")  # Invalid hex characters
except binascii.Error as e:
    assert "Non-hexadecimal digit" in str(e), f"unexpected message: {e!r}"
    print(f"✓ Correctly caught hex conversion failure ({type(e).__name__})")
else:
    raise AssertionError(f"hex_to_bin('xyz123') should have failed, returned {invalid_result!r}")

print("\nGitDB error handling tests completed successfully!")